## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [4]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


## <font color='blue'> WRITE CODE BELOW </font>

In [13]:
def PA1(user_reviews_path, products_path):
    blocksize = "128MB"

    reviews = dd.read_csv(
        user_reviews_path,
        blocksize=blocksize,
        dtype={"asin": "object", "reviewerID": "object"},
    )

    products = dd.read_csv(
        products_path,
        blocksize=blocksize,
        dtype={"asin": "object"},
    )

    products["price"] = dd.to_numeric(products["price"], errors="coerce")
    reviews["asin"] = reviews["asin"].astype("object")
    products["asin"] = products["asin"].astype("object")

    # Persist because both tables are reused many times
    # reviews = reviews.persist()
    products = products.persist()

    # Q1 and Q2
    ans1_dask = reviews.isna().mean() * 100
    ans2_dask = products.isna().mean() * 100

    # Q3
    avg_rating = reviews.groupby("asin")["overall"].mean(split_out=32).reset_index()

    price_rating = products[["asin", "price"]].merge(
        avg_rating,
        on="asin",
        how="inner",
        shuffle_method="p2p",
    ).dropna(subset=["price", "overall"])

    ans3_dask = price_rating["price"].corr(price_rating["overall"])

    # Q4
    price = products["price"].dropna()

    q4_mean = price.mean()
    q4_std = price.std()
    q4_median = price.quantile(0.5)
    q4_min = price.min()
    q4_max = price.max()

    # Q5
    def get_super_category(x):
        try:
            cats = ast.literal_eval(x)
            return cats[0][0]
        except Exception:
            return np.nan

    supercats = products["categories"].map(
        get_super_category,
        meta=("categories", "object"),
    )

    ans5_dask = supercats.value_counts(split_out=32)

    # Compute Q1-Q5 together
    (
        ans1_result,
        ans2_result,
        ans3_result,
        q4_mean_result,
        q4_std_result,
        q4_median_result,
        q4_min_result,
        q4_max_result,
        ans5_result,
    ) = dask.compute(
        ans1_dask,
        ans2_dask,
        ans3_dask,
        q4_mean,
        q4_std,
        q4_median,
        q4_min,
        q4_max,
        ans5_dask,
    )

    ans1 = ans1_result.to_dict()
    ans2 = ans2_result.to_dict()
    ans3 = float(ans3_result)

    ans4 = {
        "mean": float(q4_mean_result),
        "std": float(q4_std_result),
        "median": float(q4_median_result),
        "min": float(q4_min_result),
        "max": float(q4_max_result),
    }

    ans5 = ans5_result.to_dict()

    # Code to get answer 6
    product_asins_df = (
        products[["asin"]]
        .dropna()
        .drop_duplicates(split_out=32)
    )
    
    review_asins_df = (
        reviews[["asin"]]
        .dropna()
        .drop_duplicates(split_out=32)
    )
    
    missing_review_asins = review_asins_df.merge(
        product_asins_df,
        on="asin",
        how="left",
        indicator=True,
        shuffle_method="p2p",
    )
    
    missing_review_asins = missing_review_asins[
        missing_review_asins["_merge"] == "left_only"
    ]
    
    ans6 = int(len(missing_review_asins.head(1, npartitions=-1)) > 0)
    
    
    # Code to get answer 7
    def extract_related_asins(related_str):
        if not isinstance(related_str, str):
            return []
    
        try:
            related_dict = ast.literal_eval(related_str)
        except Exception:
            return []
    
        if not isinstance(related_dict, dict):
            return []
    
        asins = []
        for value in related_dict.values():
            if isinstance(value, list):
                asins.extend(value)
    
        return asins
    
    
    related_asins = products["related"].dropna().map(
        extract_related_asins,
        meta=("related", "object"),
    )
    
    related_asins_df = (
        related_asins
        .explode()
        .dropna()
        .to_frame(name="asin")
        .drop_duplicates(split_out=32)
    )
    
    missing_related_asins = related_asins_df.merge(
        product_asins_df,
        on="asin",
        how="left",
        indicator=True,
        shuffle_method="p2p",
    )
    
    missing_related_asins = missing_related_asins[
        missing_related_asins["_merge"] == "left_only"
    ]
    
    ans7 = int(len(missing_related_asins.head(1, npartitions=-1)) > 0)

    return ans1, ans2, ans3, ans4, ans5, ans6, ans7

## <font color='red'> DO NOT MODIFY </font>

In [14]:
start = time.time()
ans1, ans2, ans3, ans4, ans5, ans6, ans7 = PA1(user_reviews_path="user_reviews.csv", products_path="products.csv")
end = time.time()
print(f"execution time = {end-start}s")

KeyboardInterrupt: 

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         